# Exp1 Centralized — EfficientNetB0
**Project:** Noise-Aware Federated Learning for Robust Diabetic Retinopathy Classification
**Model:** EfficientNetB0 (ImageNet pretrained)
**Paradigm:** Centralized — semua dataset di-pool, train sekali

| Config | Value |
|--------|-------|
| Datasets | DDR + EyePACS + APTOS |
| Split | 70:15:15 stratified per-client |
| Noise aug (training) | ❌ None (clean only) |
| Test | Clean + Noisy (seed=123, identik di 3 notebook) |

> ⚠️ **JANGAN ubah `SEED` dan `NOISY_TEST_SEED`** — harus konsisten di 3 notebook.

## 1. GPU Check

In [ ]:
!nvidia-smi

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Imports

In [ ]:
import os, json, random, time, gc, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    confusion_matrix, accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score, roc_curve, auc, cohen_kappa_score
)
from sklearn.preprocessing import label_binarize

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.applications import EfficientNetB0

print("TF:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

## 4. Configuration

In [ ]:
# ── REPRODUCIBILITY (sama di 3 notebook) ─────────────────────────
SEED             = 42
NOISY_TEST_SEED  = 123
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# ── MODEL & EXPERIMENT ────────────────────────────────────────────
MODEL_NAME       = "EfficientNetB0"
EXPERIMENT_NAME  = "exp1_centralized"

# ── TRAINING ──────────────────────────────────────────────────────
IMG_SIZE         = (224, 224)
NUM_CLASSES      = 5
BATCH_SIZE       = 32
LR               = 1e-4
UNFREEZE_N       = 10
DROPOUT          = 0.4
L2               = 1e-4
EPOCHS_S1        = 20
EPOCHS_S2        = 20
CLIENT_NAMES     = ["DDR","EyePACS","APTOS"]

# ── NOISE PARAMS ──────────────────────────────────────────────────
NS_MIN, NS_MAX   = 0.01, 0.05
BL_MIN, BL_MAX   = 1.0,  3.0
BR_DELTA         = 0.2
CT_LOW, CT_HIGH  = 0.7,  1.3

CLASS_NAMES  = ["No DR","Mild","Moderate","Severe","Proliferative"]

# ── PATHS ─────────────────────────────────────────────────────────
PROJECT  = "/content/drive/MyDrive/Binus/Semester_4/Research_Methodology"
DS_BASE  = "/content/drive/MyDrive/S-Class/Orion/OrionFL"

APTOS_NPZ  = f"{DS_BASE}/APTOS_2019/preprocessed/orion_dr_224.npz"
DDR_NPZ    = f"{DS_BASE}/DDR_Dataset/DDR_dataset_224.npz"
EYE_NPZ    = f"{DS_BASE}/EyePACS_Dataset/training/EyePACS_dataset_224.npz"
EYE_CSV    = f"{DS_BASE}/EyePACS_Dataset/training/trainLabels.csv"

RES      = f"{PROJECT}/Result/{MODEL_NAME}/{EXPERIMENT_NAME}"
MDL_DIR  = f"{RES}/models"
LOG_DIR  = f"{RES}/logs"
FIG_DIR  = f"{RES}/figures"
for d in [MDL_DIR, LOG_DIR, FIG_DIR]: os.makedirs(d, exist_ok=True)

print(f"Model      : {MODEL_NAME}")
print(f"Experiment : {EXPERIMENT_NAME}")
print(f"Results    : {RES}")

## 5. Load Datasets

In [ ]:
def to01(X):
    X = X.astype(np.float32)
    return X / 255.0 if X.max() > 1.0 else X

print("Loading DDR...")
d = np.load(DDR_NPZ, allow_pickle=True)
X_ddr, y_ddr = to01(d["images"]), d["labels"].astype(np.int64)
print(f"  {X_ddr.shape} | classes: {np.unique(y_ddr)}")

print("Loading EyePACS...")
d = np.load(EYE_NPZ, allow_pickle=True)
X_eye = to01(d["images"])
y_eye = pd.read_csv(EYE_CSV)["level"].values.astype(np.int64)
assert len(X_eye)==len(y_eye), f"Mismatch: {len(X_eye)} vs {len(y_eye)}"
print(f"  {X_eye.shape} | classes: {np.unique(y_eye)}")

print("Loading APTOS...")
d = np.load(APTOS_NPZ, allow_pickle=True)
X_apt, y_apt = to01(d["images"]), d["labels"].astype(np.int64)
print(f"  {X_apt.shape} | classes: {np.unique(y_apt)}")

## 6. Per-Client Split 70:15:15

In [ ]:
def split_dataset(X, y, seed=SEED):
    X_tr,X_tmp,y_tr,y_tmp = train_test_split(X,y,test_size=.30,random_state=seed,stratify=y)
    X_v,X_te,y_v,y_te     = train_test_split(X_tmp,y_tmp,test_size=.50,random_state=seed,stratify=y_tmp)
    return X_tr,X_v,X_te,y_tr,y_v,y_te

Xtr_d,Xv_d,Xte_d,ytr_d,yv_d,yte_d = split_dataset(X_ddr, y_ddr)
Xtr_e,Xv_e,Xte_e,ytr_e,yv_e,yte_e = split_dataset(X_eye, y_eye)
Xtr_a,Xv_a,Xte_a,ytr_a,yv_a,yte_a = split_dataset(X_apt, y_apt)
print(f"DDR     train:{len(ytr_d):>6,}  val:{len(yv_d):>5,}  test:{len(yte_d):>5,}")
print(f"EyePACS train:{len(ytr_e):>6,}  val:{len(yv_e):>5,}  test:{len(yte_e):>5,}")
print(f"APTOS   train:{len(ytr_a):>6,}  val:{len(yv_a):>5,}  test:{len(yte_a):>5,}")

# Pool untuk centralized
rng = np.random.default_rng(SEED)
X_train = np.concatenate([Xtr_d, Xtr_e, Xtr_a])
y_train = np.concatenate([ytr_d, ytr_e, ytr_a])
idx = rng.permutation(len(y_train))
X_train, y_train = X_train[idx], y_train[idx]
X_val = np.concatenate([Xv_d, Xv_e, Xv_a])
y_val = np.concatenate([yv_d, yv_e, yv_a])
X_test  = np.concatenate([Xte_d, Xte_e, Xte_a])
y_test  = np.concatenate([yte_d, yte_e, yte_a])
src_test = np.array(["DDR"]*len(yte_d)+["EyePACS"]*len(yte_e)+["APTOS"]*len(yte_a))
print(f"Pooled → train:{len(y_train):,}  val:{len(y_val):,}  test:{len(y_test):,}")
del X_ddr, X_eye, X_apt; gc.collect()

## 7. Class Weights

In [ ]:
classes = np.unique(y_train)
cw_arr  = compute_class_weight("balanced", classes=classes, y=y_train)
class_weights = {int(c): float(w) for c, w in zip(classes, cw_arr)}
for c in range(NUM_CLASSES): class_weights.setdefault(c, 1.0)
print("Class weights (pooled train):")
for c in range(NUM_CLASSES):
    print(f"  {c} {CLASS_NAMES[c]:<15}: n={(y_train==c).sum():>6,}  w={class_weights[c]:.4f}")

## 8. Noise Pipeline

In [ ]:
def gauss_noise(img,sigma): return np.clip(img+np.random.normal(0,sigma,img.shape).astype(np.float32),0,1)
def gauss_blur(img,sigma): return np.clip(np.stack([gaussian_filter(img[:,:,c],sigma) for c in range(3)],-1).astype(np.float32),0,1)
def bright_contrast(img,bd,cf): m=img.mean(); return np.clip(((img-m)*cf+m+bd).astype(np.float32),0,1)

def deterministic_degrade(img, rng):
    img = img.copy()
    img = gauss_noise(img,     rng.uniform(NS_MIN,NS_MAX))
    img = gauss_blur(img,      rng.uniform(BL_MIN,BL_MAX))
    img = bright_contrast(img, rng.uniform(-BR_DELTA,BR_DELTA), rng.uniform(CT_LOW,CT_HIGH))
    return img

def make_noisy_set(X, seed=NOISY_TEST_SEED):
    print(f"  Generating noisy test set (n={len(X)}, seed={seed})...")
    rng = np.random.default_rng(seed); out = np.empty_like(X)
    for i in range(len(X)): out[i] = deterministic_degrade(X[i], rng)
    print(f"  Done. range=[{out.min():.3f},{out.max():.3f}]")
    return out

X_test_noisy = make_noisy_set(X_test)

## 9. Build EfficientNetB0

> **Catatan EfficientNetB0 vs MobileNetV2:**
> EfficientNetB0 sudah include rescaling internal (0-255 → normalized),
> jadi input `[0,1]` tetap aman karena kita normalize manual sebelumnya.
> `training=False` di inference mode penting untuk BatchNorm consistency.

In [ ]:
def build_model():
    base = EfficientNetB0(include_top=False, weights="imagenet", input_shape=(224,224,3))
    base.trainable = False
    inp = keras.Input((224,224,3))
    x   = base(inp, training=False)
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.Dropout(DROPOUT)(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax",
                       kernel_regularizer=regularizers.l2(L2))(x)
    return keras.Model(inp, out), base

model, base = build_model()
model.summary(line_length=90)
print(f"\nTotal params   : {model.count_params():,}")
print(f"Trainable (S1) : {sum(tf.size(w).numpy() for w in model.trainable_weights):,}")

## 10. TF Datasets

In [ ]:
AUT = tf.data.AUTOTUNE
def make_ds(X, y, shuffle=False, seed=None):
    ds = tf.data.Dataset.from_tensor_slices((X,y))
    if shuffle: ds = ds.shuffle(len(X), seed=seed)
    return ds.batch(BATCH_SIZE).prefetch(AUT)
train_ds   = make_ds(X_train, y_train, shuffle=True, seed=SEED)
val_ds     = make_ds(X_val, y_val)
test_clean = make_ds(X_test, y_test)
test_noisy = make_ds(X_test_noisy, y_test)
print(f"train:{len(train_ds)}  val:{len(val_ds)}  test_c:{len(test_clean)}  test_n:{len(test_noisy)}")

## 11. Stage 1 — Frozen Backbone (20 epochs)

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(LR),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"])

s1_ckpt = f"{MDL_DIR}/{EXPERIMENT_NAME}_s1_best.keras"
cbs_s1  = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=3, min_lr=1e-7, verbose=1),
    keras.callbacks.ModelCheckpoint(s1_ckpt, monitor="val_loss", save_best_only=True, verbose=0),
]
print(f"Stage 1: {EPOCHS_S1} epochs (frozen backbone)")
t0 = time.time()
h1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_S1,
               class_weight=class_weights, callbacks=cbs_s1, verbose=1)
print(f"Done in {(time.time()-t0)/60:.1f} min")

## 12. Stage 2 — Fine-Tune Top-10 Layers (20 epochs)

In [ ]:
base.trainable = True
for layer in base.layers[:-UNFREEZE_N]: layer.trainable = False
for layer in base.layers[-UNFREEZE_N:]: layer.trainable = True
for layer in base.layers:
    if isinstance(layer, layers.BatchNormalization): layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(LR),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"])

s2_ckpt = f"{MDL_DIR}/{EXPERIMENT_NAME}_s2_best.keras"
cbs_s2  = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=3, min_lr=1e-7, verbose=1),
    keras.callbacks.ModelCheckpoint(s2_ckpt, monitor="val_loss", save_best_only=True, verbose=0),
]
print(f"Stage 2: {EPOCHS_S2} epochs (top-{UNFREEZE_N} unfrozen, BN frozen)")
t0 = time.time()
h2 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_S2,
               class_weight=class_weights, callbacks=cbs_s2, verbose=1)
print(f"Done in {(time.time()-t0)/60:.1f} min")

## 13. Save Model + Training Curves

In [ ]:
final_path = f"{MDL_DIR}/{EXPERIMENT_NAME}_final.keras"
model.save(final_path); print(f"Saved: {final_path}")

s1_len = len(h1.history["loss"])
ep     = list(range(1, s1_len + len(h2.history["loss"]) + 1))
hist   = {"loss":         h1.history["loss"]         + h2.history["loss"],
          "val_loss":     h1.history["val_loss"]     + h2.history["val_loss"],
          "accuracy":     h1.history["accuracy"]     + h2.history["accuracy"],
          "val_accuracy": h1.history["val_accuracy"] + h2.history["val_accuracy"],
          "stage": ["s1"]*s1_len + ["s2"]*len(h2.history["loss"])}
pd.DataFrame(hist).assign(epoch=ep).to_csv(f"{LOG_DIR}/training_history.csv", index=False)

fig, ax = plt.subplots(1,2,figsize=(13,5))
ax[0].plot(ep,hist["accuracy"],label="Train"); ax[0].plot(ep,hist["val_accuracy"],"--",label="Val")
ax[0].axvline(s1_len,color="gray",ls=":",label="S1→S2"); ax[0].set_title("Accuracy"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(ep,hist["loss"],label="Train"); ax[1].plot(ep,hist["val_loss"],"--",label="Val")
ax[1].axvline(s1_len,color="gray",ls=":",label="S1→S2"); ax[1].set_title("Loss"); ax[1].legend(); ax[1].grid(alpha=.3)
plt.suptitle(f"Centralized — {MODEL_NAME}"); plt.tight_layout()
plt.savefig(f"{FIG_DIR}/training_curves.png",dpi=300,bbox_inches="tight"); plt.show()

## 14. Predict Clean + Noisy

In [ ]:
print("Predicting clean...")
yp_clean = model.predict(test_clean, verbose=1); yd_clean = np.argmax(yp_clean,1)
print("Predicting noisy...")
yp_noisy = model.predict(test_noisy, verbose=1); yd_noisy = np.argmax(yp_noisy,1)

## 15. Metrics + Robustness Drop

In [ ]:
def metrics(yt,yp,yprob):
    a = accuracy_score(yt,yp)
    p,r,f,_ = precision_recall_fscore_support(yt,yp,average="macro",zero_division=0)
    yb = label_binarize(yt,classes=np.arange(NUM_CLASSES))
    try: au = roc_auc_score(yb,yprob,multi_class="ovr",average="macro")
    except: au = float("nan")
    q = cohen_kappa_score(yt,yp,weights="quadratic")
    return dict(accuracy=float(a),precision_macro=float(p),recall_macro=float(r),
                f1_macro=float(f),auc_roc=float(au),qwk=float(q))

MKEYS = ["accuracy","precision_macro","recall_macro","f1_macro","auc_roc","qwk"]
mc = metrics(y_test,yd_clean,yp_clean)
mn = metrics(y_test,yd_noisy,yp_noisy)
print(f"{'Metric':<18} {'Clean':>8} {'Noisy':>8} {'Drop':>8}"); print("-"*44)
rob = {}
for k in MKEYS:
    d=mc[k]-mn[k]; rob[f"drop_{k}"]=float(d)
    print(f"{k:<18} {mc[k]:>8.4f} {mn[k]:>8.4f} {d:>+8.4f}")

## 16. Per-Class Metrics

In [ ]:
def per_class(yt,yp):
    p,r,f,s = precision_recall_fscore_support(yt,yp,labels=np.arange(NUM_CLASSES),zero_division=0)
    return pd.DataFrame({"class":np.arange(NUM_CLASSES),"name":CLASS_NAMES,
                          "precision":p,"recall":r,"f1":f,"support":s})
pc_c=per_class(y_test,yd_clean); pc_n=per_class(y_test,yd_noisy)
print("CLEAN:"); print(pc_c.to_string(index=False))
print("\nNOISY:"); print(pc_n.to_string(index=False))
pc_c.to_csv(f"{LOG_DIR}/per_class_clean.csv",index=False)
pc_n.to_csv(f"{LOG_DIR}/per_class_noisy.csv",index=False)

## 17. Per-Client Breakdown

In [ ]:
print(f"{'Client':<10} {'N':>5}  {'Acc_C':>7} {'Acc_N':>7} {'Drop':>7}  {'F1_C':>6} {'F1_N':>6}  {'QWK_C':>7} {'QWK_N':>7}")
print("-"*78)
pcc = {}
for cl in CLIENT_NAMES:
    m = src_test==cl
    if not m.any(): continue
    mc_=metrics(y_test[m],yd_clean[m],yp_clean[m])
    mn_=metrics(y_test[m],yd_noisy[m],yp_noisy[m])
    pcc[cl]={"n":int(m.sum()),"clean":mc_,"noisy":mn_}
    print(f"{cl:<10} {m.sum():>5,}  {mc_['accuracy']:>7.4f} {mn_['accuracy']:>7.4f} {mc_['accuracy']-mn_['accuracy']:>+7.4f}"
          f"  {mc_['f1_macro']:>6.4f} {mn_['f1_macro']:>6.4f}  {mc_['qwk']:>7.4f} {mn_['qwk']:>7.4f}")

## 18. Confusion Matrices

In [ ]:
SN=["NoDR","Mild","Mod.","Sev.","Prol."]
model_ref = model if 'model' in dir() else gm
cm_c=confusion_matrix(y_test,yd_clean,labels=np.arange(NUM_CLASSES))
cm_n=confusion_matrix(y_test,yd_noisy,labels=np.arange(NUM_CLASSES))
fig,ax=plt.subplots(1,2,figsize=(13,6))
for a,cm,title in [(ax[0],cm_c,"Clean"),(ax[1],cm_n,"Noisy")]:
    im=a.imshow(cm,cmap="Blues"); plt.colorbar(im,ax=a,fraction=.046)
    a.set_xticks(range(NUM_CLASSES)); a.set_yticks(range(NUM_CLASSES))
    a.set_xticklabels(SN,rotation=30,ha="right"); a.set_yticklabels(SN)
    th=cm.max()/2
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            a.text(j,i,str(cm[i,j]),ha="center",va="center",fontsize=9,color="white" if cm[i,j]>th else "black")
    a.set_xlabel("Predicted"); a.set_ylabel("True"); a.set_title(f"{EXPERIMENT_NAME} — {title}")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/confusion_matrices.png",dpi=300,bbox_inches="tight"); plt.show()

## 19. ROC Curves

In [ ]:
yb=label_binarize(y_test,classes=np.arange(NUM_CLASSES))
COLS=plt.cm.tab10(np.linspace(0,1,NUM_CLASSES))
fig,ax=plt.subplots(1,2,figsize=(13,6))
for a,yprob,title in [(ax[0],yp_clean,"Clean"),(ax[1],yp_noisy,"Noisy")]:
    for i in range(NUM_CLASSES):
        fpr,tpr,_=roc_curve(yb[:,i],yprob[:,i])
        a.plot(fpr,tpr,color=COLS[i],lw=1.8,label=f"{CLASS_NAMES[i]} (AUC={auc(fpr,tpr):.3f})")
    a.plot([0,1],[0,1],"k--",lw=1); a.set_xlabel("FPR"); a.set_ylabel("TPR")
    a.set_title(f"{EXPERIMENT_NAME} ROC — {title}"); a.legend(fontsize=9,loc="lower right"); a.grid(alpha=.3)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/roc_curves.png",dpi=300,bbox_inches="tight"); plt.show()

## 20. Robustness Drop Visualization

In [ ]:
MLABELS=["Accuracy","Precision","Recall","F1","AUC-ROC","QWK"]
cv=[mc[k] for k in MKEYS]; nv=[mn[k] for k in MKEYS]; dv=[rob[f"drop_{k}"] for k in MKEYS]
x=np.arange(len(MLABELS)); w=.35
fig,ax=plt.subplots(1,2,figsize=(13,5))
ax[0].bar(x-w/2,cv,w,label="Clean",color="#2196F3",alpha=.85)
ax[0].bar(x+w/2,nv,w,label="Noisy",color="#E53935",alpha=.85)
for i,(a_,b_) in enumerate(zip(cv,nv)):
    ax[0].text(i-w/2,a_+.01,f"{a_:.3f}",ha="center",fontsize=8)
    ax[0].text(i+w/2,b_+.01,f"{b_:.3f}",ha="center",fontsize=8)
ax[0].set_xticks(x); ax[0].set_xticklabels(MLABELS); ax[0].set_ylim(0,1.1)
ax[0].set_title("Clean vs Noisy"); ax[0].legend(); ax[0].grid(axis="y",alpha=.3)
clrs=["#4CAF50" if d<=.05 else "#FFC107" if d<=.1 else "#F44336" for d in dv]
ax[1].bar(x,dv,color=clrs,alpha=.85,edgecolor="black",lw=.5)
for i,d in enumerate(dv): ax[1].text(i,d+.003,f"{d:+.3f}",ha="center",fontsize=9,fontweight="bold")
ax[1].set_xticks(x); ax[1].set_xticklabels(MLABELS)
ax[1].set_title("Robustness Drop (↓ = lebih robust)"); ax[1].axhline(0,color="k",lw=.8,ls="--"); ax[1].grid(axis="y",alpha=.3)
plt.suptitle(f"{EXPERIMENT_NAME} — {MODEL_NAME}"); plt.tight_layout()
plt.savefig(f"{FIG_DIR}/robustness_drop.png",dpi=300,bbox_inches="tight"); plt.show()

## 21. Save Results + Summary

In [ ]:
res={
    "experiment":EXPERIMENT_NAME,"model":MODEL_NAME,"paradigm":"Centralized",
    
    "n_train":int(len(y_train)),"n_val":int(len(y_val)),
    "n_test":int(len(y_test)),
    "metrics_clean":mc,"metrics_noisy":mn,"robustness_drop":rob,"per_client":pcc,
    "confusion_matrix_clean":cm_c.tolist(),"confusion_matrix_noisy":cm_n.tolist(),
    "config":dict(seed=SEED,noisy_test_seed=NOISY_TEST_SEED,batch=BATCH_SIZE,lr=LR,
                  "epochs_s1":EPOCHS_S1,"epochs_s2":EPOCHS_S2,unfreeze_n=UNFREEZE_N)
}
with open(f"{LOG_DIR}/results.json","w") as f: json.dump(res,f,indent=2)
row={"experiment":EXPERIMENT_NAME,"paradigm":"Centralized","model":MODEL_NAME}
row.update({f"clean_{k}":mc[k] for k in MKEYS})
row.update({f"noisy_{k}":mn[k] for k in MKEYS})
row.update(rob)
pd.DataFrame([row]).to_csv(f"{LOG_DIR}/summary_row.csv",index=False)
print(f"Saved to: {RES}")
print("\n"+"="*60)
print(f"  EXP1 CENTRALIZED — DONE")
print("="*60)
print(f"  CLEAN  | Acc:{mc['accuracy']:.4f}  F1:{mc['f1_macro']:.4f}  AUC:{mc['auc_roc']:.4f}  QWK:{mc['qwk']:.4f}")
print(f"  NOISY  | Acc:{mn['accuracy']:.4f}  F1:{mn['f1_macro']:.4f}  AUC:{mn['auc_roc']:.4f}  QWK:{mn['qwk']:.4f}")
print(f"  DROP   | Acc:{rob['drop_accuracy']:+.4f}  F1:{rob['drop_f1_macro']:+.4f}  AUC:{rob['drop_auc_roc']:+.4f}  QWK:{rob['drop_qwk']:+.4f}")
print("="*60)